## 6.3 – More Sorting
### Comparing Objects
Remember in the last section we mentioned the fact that our implementation of selection sort was *unstable* – for objects with equal value they are not guaranteed to remain in the same order. 

In this selection we'll introduce *insertion sort*, which has a stable implementation. But first, the issue of stability highlights the fact that there can be more to an object than just a simple value. This makes sense if we think about it within the object oriented programming framework – objects have many attributes and methods, for any two given objects it might not be obvious how to *order* them. In fact, it might not even be obvious whether two objects are even *equal*!

So here's a nice collection of Python features. You can actually define these things for your own classes. If you try to use `<` with two objects, you get an error:

In [1]:
class PlayingCard:
    def __init__(self, number, suit):
        self.number = number
        self.suit = suit
        
nine_of_hearts = PlayingCard(9, "♥")
nine_of_spades = PlayingCard(9, "♠")

nine_of_hearts < nine_of_spades

TypeError: '<' not supported between instances of 'PlayingCard' and 'PlayingCard'

In addition, you can use `==` on these objects, but it will check whether the two objects are *literally* the same object. Remember objects are mutable, so if we create two instances which happen to have the same contents, they are still two separate instances in memory – they must be, we might change one and the other should remain the same. But the code below looks a bit odd:

In [2]:
nine_of_hearts1 = PlayingCard(9, "♥")
nine_of_hearts2 = PlayingCard(9, "♥")

nine_of_hearts1 == nine_of_hearts2

False

Thankfully Python has an easy way to support custom behaviour for these operators. Perhaps we are playing a card game where suit does not matter, so the cards should just be compared on value.

First of all, we can override the `__eq__` method to define our own notion of equality. This will change the behaviour of `==`, but also affect code that uses keywords like `in`, e.g. `if card in hand_of_cards`. 

It's worth noting that if you do choose to override `__eq__` you *should* also override `__hash__`. We'll come back to *hash functions* in next week's material to learn why, but the simple answer is that sets and dictionaries will break for your objects if you do not. For now you can [read more here](https://docs.python.org/3/reference/datamodel.html#object.__hash__), but I am going to be lazy and only override `__eq__` until we learn more about what hash functions do next week.

In addition, we can implement a method called `__lt__` (less than) to enable the `<` operator:

In [2]:
class PlayingCard:
    def __init__(self, number, suit):
        self.number = number
        self.suit = suit
        
    def __eq__(self, other):
        if type(other) is type(self):                   # type check to ensure they are both the same type in this case, <class 'PlayingCard'>
            return self.number == other.number          # self.number == other.number is an expression that evaluates to True or False, and then return hands that boolean back to whoever called __eq__.
        else:
            return NotImplemented
        
    def __lt__(self, other):
        if type(other) is type(self):                   # type check to ensure they are both the same type in this case, <class 'PlayingCard'>
            return self.number < other.number           # self.number < other.number is an expression that evaluates to True or False, and then return hands that boolean back to whoever called __lt__.
        else:
            return NotImplemented
        
    def __str__(self):
        return f"{self.number}{self.suit}"                   # 9♥  (friendly for a user in print)
    
    def __repr__(self):
        return f"PlayingCard({self.number}, '{self.suit}')"  # PlayingCard(9, '♥')  (unambiguous for developers, )
        
        
nine_of_hearts = PlayingCard(9, "♥")
nine_of_spades = PlayingCard(9, "♠")

print(nine_of_hearts < nine_of_spades)
print(nine_of_spades < nine_of_hearts)

False
False


In [6]:
# When you're working interactively in the Python interpreter (or a Jupyter notebook), 
# if you just type a variable name without print(), Python uses __repr__ to display it:

nine_of_hearts        # no print() — uses __repr__




PlayingCard(9, '♥')

In [5]:
print(nine_of_hearts) # uses __str__

9♥


In [ ]:
# when you put objects inside a list (or dict, tuple etc.) and print the list, Python calls __repr__ on each item, not __str__
# The reasoning is that when displaying a container, 
# Python assumes you're in a developer/debugging context and wants each item to be unambiguous rather than pretty.

hand = [nine_of_hearts, nine_of_spades]                 # list

print(nine_of_hearts)  # __str__  → 9♥
print(hand)            # __repr__ on each card → [9♥, 9♠]

9♥
[PlayingCard(9, '♥'), PlayingCard(9, '♠')]


In [4]:
nine_of_hearts1 = PlayingCard(9, "♥")
nine_of_hearts2 = PlayingCard(9, "♥")

nine_of_hearts1 == nine_of_hearts2

True

Notice if we want the old behaviour which compares whether the two objects are identical, we can do that with the `is` operator:

In [5]:
nine_of_hearts1 is nine_of_hearts2

False

Also worth explaining `__str__` and `__repr__`. These are called when Python wants a string based representation of the object. `str` is for a human-readable version of the string – called if you print the object. `repr` is supposed to give an unambiguous representation, it will be called by debuggers for example, but it also gets called when we put multiple items in a list and print the list, so it's helpful for us to define both.

In [8]:
print(PlayingCard(9, "♥"))
print([PlayingCard(9, "♠"), PlayingCard(9, "♥")])

9♥
[PlayingCard(9, '♠'), PlayingCard(9, '♥')]


In [ ]:
# when you define __eq__, Python automatically sets __hash__ to None, 
# because it detects a contradiction: if two objects can be equal but have different hashes, sets and dicts break.

nine_of_hearts = PlayingCard(9, "♥")
nine_of_spades = PlayingCard(9, "♠")

nine_of_hearts == nine_of_spades  # True (your __eq__ says so)


True

In [12]:
# But if hash is broken:
hash(nine_of_hearts)  # TypeError: unhashable type: 'PlayingCard'


TypeError: unhashable type: 'PlayingCard'

In [13]:
# So this would fail too:
hand = {nine_of_hearts, nine_of_spades}  # can't put them in a set

TypeError: unhashable type: 'PlayingCard'

### hash explained

- Python's rule is: if two objects are equal, they must have the same hash. 
- Sets and dicts use the hash first to find a "bucket", then use __eq__ to confirm. If equal objects have different hashes, Python would look in the wrong bucket and never find them — silent broken behaviour.

The fix: 

```python
def __hash__(self):
    return hash(self.number)  # equal cards have same number → same hash ✓
```

In [15]:
class PlayingCard:
    def __init__(self, number, suit):
        self.number = number
        self.suit = suit

    def __hash__(self):
        return hash(self.number)                        # equal cards have same number → same hash ✓
        
    def __eq__(self, other):
        if type(other) is type(self):                   # type check to ensure they are both the same type in this case, <class 'PlayingCard'>
            return self.number == other.number          # self.number == other.number is an expression that evaluates to True or False, and then return hands that boolean back to whoever called __eq__.
        else:
            return NotImplemented
        
    def __lt__(self, other):
        if type(other) is type(self):                   # type check to ensure they are both the same type in this case, <class 'PlayingCard'>
            return self.number < other.number           # self.number < other.number is an expression that evaluates to True or False, and then return hands that boolean back to whoever called __lt__.
        else:
            return NotImplemented
        
    def __str__(self):
        return f"{self.number}{self.suit}"                   # 9♥  (friendly for a user in print)
    
    def __repr__(self):
        return f"PlayingCard({self.number}, '{self.suit}')"  # PlayingCard(9, '♥')  (unambiguous for developers, )
        
        
nine_of_hearts = PlayingCard(9, "♥")
nine_of_spades = PlayingCard(9, "♠")

print(nine_of_hearts < nine_of_spades)
print(nine_of_spades < nine_of_hearts)

False
False


In [ ]:
# now this should work:

nine_of_hearts == nine_of_spades  # True
hash(nine_of_hearts) == hash(nine_of_spades)  # True ✓ consistent

hand = {nine_of_hearts, nine_of_spades}  # works — and only keeps one, since they're "equal"

print(hand)          # see what's in the set, it keeps whichever was inserted first — nine_of_hearts in this case

{PlayingCard(9, '♥')}


## functools
And as a final note, we obviously haven't implemented any of the other operators, and there are corresponding methods like `__gt__` for `>` or `__le__` for `<=`. It is quite a bore to go through and implement them all, so you can use a decorator from the `functools` module to help [if you want](https://docs.python.org/3/library/functools.html#functools.total_ordering). 

In [18]:
from functools import total_ordering

@total_ordering                                         # The decorator works out the missing methods logically — if a < b is defined, then a > b is just b < a, and so on.
class PlayingCard:
    def __init__(self, number, suit):
        self.number = number
        self.suit = suit

    def __hash__(self):
        return hash(self.number)                        # equal cards have same number → same hash ✓
        
    def __eq__(self, other):
        if type(other) is type(self):                   # type check to ensure they are both the same type in this case, <class 'PlayingCard'>
            return self.number == other.number          # self.number == other.number is an expression that evaluates to True or False, and then return hands that boolean back to whoever called __eq__.
        else:
            return NotImplemented
        
    def __lt__(self, other):
        if type(other) is type(self):                   # type check to ensure they are both the same type in this case, <class 'PlayingCard'>
            return self.number < other.number           # self.number < other.number is an expression that evaluates to True or False, and then return hands that boolean back to whoever called __lt__.
        else:
            return NotImplemented
        
    def __str__(self):
        return f"{self.number}{self.suit}"                   # 9♥  (friendly for a user in print)
    
    def __repr__(self):
        return f"PlayingCard({self.number}, '{self.suit}')"  # PlayingCard(9, '♥')  (unambiguous for developers, )
        
        
nine_of_hearts = PlayingCard(9, "♥")
nine_of_spades = PlayingCard(9, "♠")

print(nine_of_hearts < nine_of_spades)
print(nine_of_spades < nine_of_hearts)

False
False


In [19]:
# now we get all the other operators for free because we applied the @total_ordering decorator to the class:

nine_of_hearts = PlayingCard(9, "♥")
king_of_spades = PlayingCard(13, "♠")

print(nine_of_hearts < king_of_spades)   # True  — your __lt__
print(nine_of_hearts > king_of_spades)   # False — derived automatically
print(nine_of_hearts <= king_of_spades)  # True  — derived automatically
print(nine_of_hearts >= king_of_spades)  # False — derived automatically

True
False
True
False


### Insertion Sort
Now we can finally get around to the stable sort implementation. Insertion sort is actually similar to selection sort in the sense that it builds up the sorted list one element at a time. And like selection sort, while we could implement it creating a new list, it is more efficient to have it work in-place.

Here is the basic idea: at the start of iteration $k$ we assume that the sub-list of just the first $k$ elements is already sorted. Obviously this is okay because on iteration number 1 we only need to consider the first element as its own list, and any list of length 1 must be considered sorted.

Then you take the next element from the list (the item at position `k` in the list, assuming zero-indexing – so the item directly after your sorted sub-list) and you store this item in a temporary variable. Now you go backwards through the array, checking the item at position `i = k-1`, `k-2`, `k-3`, and so on. Each time, you compare the item to the temporary variable: if it is less than or equal, you stop and insert the temporary variable value into position `i+1`. If the item is bigger, you move it one item to the right in the array, and continue moving down.

We will need to test for either “less than or equal” or “greater than” in our code, but we did not implement these for the `PlayingCard` class. But we did implement `__lt__` and `__eq__`, so we can combine both in the code below to avoid having to redefine the class with new methods.

Have a look at the code below to ensure you understand!

# simplified explanation:

The big picture
- Imagine you're sorting playing cards in your hand. 
- You pick up one card at a time, and slot it into the right position among the cards you're already holding. 
- At every point, the cards in your hand are sorted — you're just growing the sorted portion one card at a time.That's insertion sort. 
- The list is split into two imaginary regions:

```python
[sorted region | unsorted region]
```
- Each iteration takes the first item from the unsorted region and inserts it into the right place in the sorted region.

#### What k is
- `k`is the **index** of the card you just picked up — the next item to insert.
- So k goes 1, 2, 3, 4... — we start at 1 because a `single element (element 0)` is **already** trivially sorted.

```python
for k in range(1, len(my_list)):


k=1:  [37 | 42, 9, 19, 35, 4, 53, 22]   ← insert 42
k=2:  [37, 42 | 9, 19, 35, 4, 53, 22]   ← insert 9
k=3:  [9, 37, 42 | 19, 35, 4, 53, 22]   ← insert 19
```
#### What temp is
- You save the item you're about to insert, because you're **going to overwrite its slot as you shift things right** to make room for it.
```python
temp = my_list[k]
```

#### What `i` is and why reversed
- `i` walks **backwards** through the **sorted region**, comparing each item to `temp`. 
- You **go backwards** because you're **looking for where temp belongs**, and **shifting items right** as you go to make a gap.
```python
for i in reversed(range(-1, k)):
```
- range(-1, k) goes from -1 up to k-1. Then reversed() flips it. So it actually **starts at k-1** and ends at -1.

```python
reversed(range(-1, k)) when k=2 gives: 1, 0, -1
```

- The `-1` is a sentinel — it means **"you've gone past the start of the list, so insert here regardless."**
- When i == -1 you insert at `i + 1` which is `0` — the start of the list.

```python 
if i == -1 or ...:
    my_list[i+1] = temp  # i+1 = -1+1 = 0
    break
```

- A `sentinel` is just a special value you add purely to signal **"we've hit a boundary"** — it's not real data, it's just a trigger.

#### Tracing k=2 step by step:

State at start: `[37, 42, 9, ...]`, `k=2`, `temp=9`

```python
i=1: my_list[1] is 42. Is 42 < 9 or 42 == 9? No → shift 42 right: [37, 42, 42, ...]
i=0: my_list[0] is 37. Is 37 < 9 or 37 == 9? No → shift 37 right: [37, 37, 42, ...]
i=-1: sentinel hit → insert temp at i+1 = position 0: [9, 37, 42, ...]
```
- The condition unpacked:
```python 
if i == -1 or (my_list[i] < temp or my_list[i] == temp):
```

This is asking: "should I stop and insert here?" — yes if:

- `i == -1` — hit the start of the list, no more room to go left, or
- `my_list[i] < temp` — found an item smaller than temp, so temp belongs just to the right, or
- `my_list[i] == temp` — found an equal item (inserting here maintains stability — equal items stay in original order)
- If none of those are true, the item at `i` is bigger than `temp`, so shift it right and keep going left.


In [7]:
def insertion_sort(my_list):
    for k in range(1, len(my_list)):
        temp = my_list[k]
        
        for i in reversed(range(-1, k)):
            if i == -1 or (my_list[i] < temp or my_list[i] == temp):
                my_list[i+1] = temp
                break
            else:
                my_list[i+1] = my_list[i]
    

my_list = [37, 42, 9, 19, 35, 4, 53, 22]
insertion_sort(my_list)
print(my_list)

[4, 9, 19, 22, 35, 37, 42, 53]


In [8]:
my_cards = [PlayingCard(9, "♠"), PlayingCard(9, "♥"), PlayingCard(4, "♣")]
insertion_sort(my_cards)
print(my_cards)

[4♣, 9♠, 9♥]


Notice the list has been sorted successfully, and the order is stable: the two 9s are in the same positions they were at the start.

***Exercise:*** go and modify the code for the PlayingCard class, using the `total_ordering` [decorator](https://docs.python.org/3/library/functools.html#functools.total_ordering) from `functools` to enable the use of `<=`, then modify the `insertion_sort` function to use that instead of the combination of `<` and `==`.

In [20]:
# updated based on exercise: 

def insertion_sort(my_list):
    for k in range(1, len(my_list)):
        temp = my_list[k]
        
        for i in reversed(range(-1, k)):
            if i == -1 or (my_list[i] <= temp):                 # ← was: my_list[i] < temp or my_list[i] == temp
                my_list[i+1] = temp
                break
            else:
                my_list[i+1] = my_list[i]
    

# Test with integers
my_list = [37, 42, 9, 19, 35, 4, 53, 22]
insertion_sort(my_list)
print(my_list)

[4, 9, 19, 22, 35, 37, 42, 53]


In [21]:
# Test with PlayingCards
cards = [
    PlayingCard(9, "♥"),
    PlayingCard(3, "♠"),
    PlayingCard(9, "♠"),   # second 9 — should stay after 9♥ (stable sort)
    PlayingCard(7, "♦"),
]
insertion_sort(cards)
print(cards)

[PlayingCard(3, '♠'), PlayingCard(7, '♦'), PlayingCard(9, '♥'), PlayingCard(9, '♠')]



### Insertion Sort Complexity
***Have a think about the time complexity of insertion sort before continuing.***

In the best case, the complexity is actually $O(n)$, which is when the list is already sorted. In this case the inner for loop will never perform more than one operation, so does not scale with the length of the list at all, so we are just left with the outer for loop for $O(n)$.

In the average and worst case, the complexity is $O(n^2)$ again. The worst case is easiest to demonstrate, and occurs when the list is in exactly the reverse order. In that case, both for loops iterate the maximum number of times. The outer for loop repeats $n$ times the inner for loop, which itself repeats $n$ times, so $O(n^2)$. The average case is better, but that inner for loop still scales with the length of the list, and we get the same quadratic complexity class.

Insertion sort outperforms selection sort in the number of comparisons, and the closer the list is to already being sorted, the better it performs.

It might actually be that the *swapping* of elements is the time-sensitive bottleneck in our system. In that case, `selection` sort actually performs best. `Insertion` sort **moves its items around more**.

### helpful analogy:

A rough analogy — imagine sorting a hand of playing cards on a table:

- Selection sort — scan all the cards, pick up the lowest, place it at the front. One pick-up per round.
- Insertion sort — take the next card and slide it left past each card it beats, one nudge at a time.

The sliding is the extra movement that adds up.

## What Next?
Once you are done, go back to Engage to move onto the next section.